# Phase 2: Feature Engineering And Train/Test Split
This is where I:
1. Turned the raw alert columns into a clean set of features a machine learning model can actually learn from
2. Split the data into a training set and a test set

In [1]:
import json
import glob
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
DATA_DIR = "./WARP_Falco_Alerts_Labeled_Dataset/Collected_Alerts/Final Processed and Labeled Alerts"
alert_json_files = sorted(glob.glob(f"{DATA_DIR}/*.json"))

all_records = []
for filepath in alert_json_files:
    with open(filepath, encoding="utf-8") as f:
        records = json.load(f)
        all_records.extend(records)

df = pd.json_normalize(all_records)
print("Loaded:", df.shape)

Loaded: (176649, 36)


## Why the we can't just feed the raw columns into a modle
Machine learning models need numbers, not free text or nested lists. Right now the data has problems like:
- `tags` is a *list* of strings (e.g. `["container", "mitre_lateral_movement"]`) and models can't read a list directly.
- `output_fields.proc.cmdline` is free text (e.g. `"bash"`, `"curl http://..."`) - Has different values, using one-hot encode  directly will be difficult
- `time` is a timestamp string which is not directly useful, but *when* something happens it can be a useful pattern.
- Several columns are almost entirely empty, however *whether a field is present at all* can itself be a signal, so we don't want to just throw that information away.

So my job here is to turn all of that into a small number of clean, useful columns.

## Extract the MITRE ATT&CK tactic from `tags`

I see that Falco tags each alert with metadata, and some of those tags start with `mitre_` (e.g. `mitre_privilege_escalation`, `mitre_persistence`). This is a useful security context, it tells us which stage of an attack this alert might relate to, according to the MITRE ATT&CK framework. So I pull the first `mitre_*` tag out into its own column, and use `"none"` for alerts that don't have one.


In [3]:
def extract_mitre_tactic(tags):
    """return MITRE ATT&CK tactic tag found. Else return 'none'."""
    if not isinstance(tags, list):
        return "none"
    for tag in tags:
        if isinstance(tag, str) and tag.startswith("mitre_"):
            return tag
    return "none"

df["mitre_tactic"] = df["tags"].apply(extract_mitre_tactic)
print(df["mitre_tactic"].value_counts())


mitre_tactic
mitre_lateral_movement        59072
mitre_persistence             55349
mitre_credential_access       21376
mitre_execution               15847
mitre_remote_access_tools      9378
mitre_exfiltration             9239
mitre_privilege_escalation     5782
none                            296
mitre_defense_evasion           208
mitre_discovery                 102
Name: count, dtype: int64


## Build simple features from the command line text
We won't do full Natural Language Processing, and we can't feed raw text directly to the model, so instead we do two things with the command line text:

Get the length of the command, because longer commands are a bit more likely to be attacks.
Check if it contains suspicious words like curl, wget, base64 —> yes (1) or no (0).

We're giving these as extra clues to the model, not a rule that decides on its own whether something is an attack. The model still looks at all the features together, priority, rule, and these two, and learns how much weight to give each one.

In [4]:
# fill in empty commands with an empty string, so we don't get errors later
cmdline_column = df["output_fields.proc.cmdline"].fillna("")

# get the length of each command
cmdline_lengths = []
for cmd in cmdline_column:
    cmdline_lengths.append(len(str(cmd)))
df["cmdline_length"] = cmdline_lengths

# list of words that often show up in attacker commands
suspicious_keywords = ["curl", "wget", "nc ", "chmod 777", "base64", "/etc/shadow", "history"]

# check each command for any of those words
def has_suspicious_keyword(cmd):
    cmd = str(cmd).lower()
    for keyword in suspicious_keywords:
        if keyword in cmd:
            return 1
    return 0

df["suspicious_cmd_flag"] = df["output_fields.proc.cmdline"].apply(has_suspicious_keyword)

sample = df[["output_fields.proc.cmdline", "cmdline_length", "suspicious_cmd_flag"]].sample(5, random_state=1)
print(sample)

# Show a few rows where the flag WAS triggered, to confirm it works
is_suspicious = df["suspicious_cmd_flag"] == 1
flagged = df[is_suspicious]
flagged = flagged[["output_fields.proc.cmdline", "cmdline_length", "suspicious_cmd_flag"]]
print(f"Total flagged commands: {len(flagged)}")
print(flagged) 

        output_fields.proc.cmdline  cmdline_length  suspicious_cmd_flag
19852       container:ddb65231654e              22                    0
134902  event-generator run --loop              26                    0
53855   event-generator run --loop              26                    0
42705       container:4f3731b65d62              22                    0
142844                       login               5                    0
Total flagged commands: 2
                               output_fields.proc.cmdline  cmdline_length  \
13783   curl -L https://github.com/docker/compose/rele...             127   
175131                                    cat /etc/shadow              15   

        suspicious_cmd_flag  
13783                     1  
175131                    1  



## Fill in gaps and add "was this field present" flags

For `user.name` and `container.image.repository`, I fill missing values with `"unknown"`. Since a small number of rows are missing these, and `"unknown"` is a very valid category for a model to learn from.

For the fields that are missing over 60% of the time (like `proc.name` and `fd.name`), rather than including all that noise directly, I turn their *presence* or *absence* into a simple 1/0 feature. This keeps things simple while still capturing the useful signal.

In [5]:
df["user_name"] = df["output_fields.user.name"].fillna("unknown")
df["image_repo"] = df["output_fields.container.image.repository"].fillna("unknown")

# 1, if the field has a value, 0 if it's missing
df["has_process_detail"] = df["output_fields.proc.name"].notna().astype(int)
df["has_file_event"] = df["output_fields.fd.name"].notna().astype(int)


## Extract the hour of day
I extract just the hour as a simple numeric feature.

In [6]:
df["hour"] = pd.to_datetime(df["time"]).dt.hour
df[["time", "hour"]].head(3)


,time,hour
0,2021-11-19T20:30:47.024077548Z,20
1,2021-11-19T20:31:16.488309915Z,20
2,2021-11-19T20:44:04.609659942Z,20


## My final feature set

Here's the full list of features I am keeping, and why each one is there:
1. `priority`: a **category** feature, representing Falco's own severity judgement for the alert.
2. `rule`: a **category** feature, representing which Falco rule fired.
3. `mitre_tactic`: a **category** feature, giving MITRE ATT&CK context when one applies to the alert.
4. `user_name`: a **category** feature, showing whether the process ran as root, bin, or an unknown user.
5. `image_repo`: a **category** feature, showing which container image raised the alert.
6. `hour`: **number** feature, capturing time-of-day as a pattern.
7. `cmdline_length`: **number** feature, since longer commands can indicate attacker activity.
8. `suspicious_cmd_flag`: **number** feature (0 or 1), showing whether the command contains a keyword commonly seen in attacks.
9. `has_process_detail`: **number** feature (0 or 1), showing whether Falco reported process detail for that alert.
10. `has_file_event`: **number** feature (0 or 1), showing whether the alert was a file-related event.

Our **target** which is what we're trying to predict, that is `label`, turned into `1` for attack and `0` for normal.

In [7]:
feature_cols = [
    "priority", "rule", "mitre_tactic",
    "user_name", "image_repo","hour", "cmdline_length", 
    "suspicious_cmd_flag", "has_process_detail", "has_file_event",
]
categorical_cols = ["priority", "rule", "mitre_tactic", "user_name", "image_repo"]
numeric_cols = ["hour", "cmdline_length", "suspicious_cmd_flag", "has_process_detail", "has_file_event"]

# X is for features which the model will learn from
# So we extract those columns into variable X
X = df[feature_cols].copy()

# y is the target we are trying to predict
y = (df["label"] == "attack").astype(int)

print("Feature matrix shape:", X.shape)
print("Attack rate in target:", round(y.mean() * 100, 3), "%")
X.head()


Feature matrix shape: (176649, 10)
Attack rate in target: 0.786 %


,priority,rule,mitre_tactic,user_name,image_repo,hour,cmdline_length,suspicious_cmd_flag,has_process_detail,has_file_event
0,Notice,Terminal shell in container,mitre_execution,root,falcosecurity/falco,20,4,0,1,0
1,Warning,Delete or rename shell history,mitre_defense_evasion,root,unknown,20,4,0,0,1
2,Notice,Launch Sensitive Mount Container,mitre_lateral_movement,root,busybox,20,22,0,0,0
3,Notice,Launch Privileged Container,mitre_lateral_movement,root,busybox,20,22,0,0,0
4,Notice,Launch Privileged Container,mitre_lateral_movement,root,busybox,20,52,0,0,0


## Numeric feature summary (mean / median / spread)

Quick check on the five numeric features before splitting: typical values and whether attacks look different from normal alerts.

In [8]:
# Overall stats (mean, median, min, max, etc.)
print(df[numeric_cols].describe())

# Average value by class
print("\nMean by label:")
print(df.groupby("label")[numeric_cols].mean())

                hour  cmdline_length  suspicious_cmd_flag  has_process_detail  \
count  176649.000000   176649.000000        176649.000000       176649.000000   
mean       11.776551       28.655220             0.000011            0.144445   
std         7.481333       16.095552             0.003365            0.351541   
min         0.000000        2.000000             0.000000            0.000000   
25%         5.000000       22.000000             0.000000            0.000000   
50%        12.000000       26.000000             0.000000            0.000000   
75%        19.000000       26.000000             0.000000            0.000000   
max        23.000000      278.000000             1.000000            1.000000   

       has_file_event  
count   176649.000000  
mean         0.333667  
std          0.471524  
min          0.000000  
25%          0.000000  
50%          0.000000  
75%          1.000000  
max          1.000000  

Mean by label:
             hour  cmdline_length  sus

**what this tells me**:
Attack and normal alerts look almost the same on `hour` and `cmdline_length`, and `suspicious_cmd_flag` but is attacks always have `has_process_detail` = 0 and `has_file_event` = 0, while many normal alerts have those fields set (~15% and ~34%). So separation in this dataset is unlikely to come from cmdline length or time alone. In this dataset, how the alert is structured separates classes more than cmdline length or time and that is why rule and priority were included as features

##  Split into training and test sets

I use `train_test_split` with **`stratify=y`** to split the data 80/20 training/test.
`stratify=y` forces both the training set and the test set to keep the *same* attack/normal ratio as the full dataset, so our evaluation later is fair and reliable.

I set aside 20% of the data purely for testing, and never let the model see it during training.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=7,
)

# print("Train set:", X_train.shape, "| attack rate:", round(y_train.mean() * 100, 3), "%")
# print("Test set: ", X_test.shape, "| attack rate:", round(y_test.mean() * 100, 3), "%")

print( "Train set:", X_train.shape, "| normal:", (y_train == 0).sum(), "| attack:", (y_train == 1).sum(), "| attack rate:", round(y_train.mean() * 100, 3),"%")
print( "Test set: ", X_test.shape, "| normal:", (y_test == 0).sum(), "| attack:", (y_test == 1).sum(), "| attack rate:", round(y_test.mean() * 100, 3), "%")

# print("X_train (features for training):")
# print(X_train.head())

# print("\ny_train (answers for training):")
# print(y_train.head())

# # Combine X_train and y_train into one table, just for viewing
# train_view = X_train.copy()
# train_view["label"] = y_train
# print(train_view.head())

# # Same for the test set
# test_view = X_test.copy()
# test_view["label"] = y_test
# print(test_view.head())

Train set: (141319, 10) | normal: 140209 | attack: 1110 | attack rate: 0.785 %
Test set:  (35330, 10) | normal: 35052 | attack: 278 | attack rate: 0.787 %


## Save Output

Save the train/test features and labels as CSV files so we can use this for other runs too.

In [10]:
import os
os.makedirs("./data", exist_ok=True)

X_train.assign(label=y_train.values).to_csv("./data/train.csv", index=False)
X_test.assign(label=y_test.values).to_csv("./data/test.csv", index=False)

print("Saved ./data/train.csv and ./data/test.csv")

Saved ./data/train.csv and ./data/test.csv
